# Exploratory Data Analysis

This notebook demonstrates basic EDA on driving sensor data.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataio import DataLoader, load_config
from src.health import SensorHealthChecker
from src.preprocessing import Preprocessor

%matplotlib inline
sns.set_style('darkgrid')

## Load Data

In [ ]:
# Load configuration
config = load_config('../configs/dataset.yml')

# Initialize loader
loader = DataLoader('../configs/dataset.yml')
dfs = loader.load_all()

print(f"Loaded {len(dfs)} files")
if dfs:
    df = dfs[0]
    print(f"First file shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

## Data Health Check

In [ ]:
# Run health check
checker = SensorHealthChecker(config)
imu_channels = ['accX', 'accY', 'accZ', 'gyroX', 'gyroY', 'gyroZ']
gps_channels = ['lat', 'lon', 'speed']

report = checker.generate_report(df, imu_channels, gps_channels, sampling_rate=100.0)

print(f"Samples: {report['n_samples']}")
print(f"Duration: {report['duration_seconds']:.1f}s")
print(f"Critical failures: {report['critical_failures']}")
print(f"\nMissingness:")
for col, pct in report['missingness'].items():
    print(f"  {col}: {pct:.2f}%")

## Visualize Sensor Data

In [ ]:
# Plot accelerometer
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

for i, channel in enumerate(['accX', 'accY', 'accZ']):
    if channel in df.columns:
        axes[i].plot(df[channel])
        axes[i].set_ylabel(channel)
        axes[i].set_title(f'Accelerometer {channel}')

axes[-1].set_xlabel('Sample')
plt.tight_layout()
plt.show()

In [ ]:
# Plot gyroscope
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

for i, channel in enumerate(['gyroX', 'gyroY', 'gyroZ']):
    if channel in df.columns:
        axes[i].plot(df[channel])
        axes[i].set_ylabel(channel)
        axes[i].set_title(f'Gyroscope {channel}')

axes[-1].set_xlabel('Sample')
plt.tight_layout()
plt.show()

## GPS Trace

In [ ]:
# Plot GPS trajectory
if 'lat' in df.columns and 'lon' in df.columns:
    plt.figure(figsize=(10, 8))
    plt.plot(df['lon'], df['lat'], 'b-', alpha=0.5)
    plt.scatter(df['lon'].iloc[0], df['lat'].iloc[0], c='g', s=100, marker='o', label='Start')
    plt.scatter(df['lon'].iloc[-1], df['lat'].iloc[-1], c='r', s=100, marker='x', label='End')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('GPS Trajectory')
    plt.legend()
    plt.grid(True)
    plt.show()

## Statistics

In [ ]:
# Summary statistics
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe()